# Fused expert MLP in Pallas

Beat three `lax.ragged_dot`s with one Pallas TPU kernel for

```
y[i] = down[g(i)] @ (swish(x[i] @ gate[g(i)]) * (x[i] @ up[g(i)]))
```

when `F > D`, then find a shape where the speedup is measurable and explain it.

Rows arrive sorted by expert, so expert `e` owns `[offsets[e], offsets[e+1])`. Router, sort and
scatter-back are out of scope.

Biases are dropped for now. `down`'s is a true epilogue you can add outside the kernel, but
`up`/`gate`'s land before the swish, so a bias-carrying kernel adds them on the accumulator tile —
a broadcast of `bias[e]`, since a tile inside a group shares one expert. Add them once the plain
version works.

In [1]:
import functools
import os

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu

DTYPE = jnp.bfloat16

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["cmr10", "DejaVu Serif"],
        "mathtext.fontset": "cm",
        "axes.formatter.use_mathtext": True,
        "axes.unicode_minus": False,  # cmr10 has no glyph for it
        "font.size": 9,
        "lines.linewidth": 1.0,
        "lines.markersize": 3.0,
        "axes.linewidth": 0.6,
        "grid.linewidth": 0.4,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "legend.frameon": False,
    }
)

print(jax.__version__, jax.devices())

0.6.2 [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


In [2]:
@functools.partial(jax.jit, static_argnums=(1, 2))
def normal(key, shape: tuple[int, ...], fan_in: int) -> jnp.ndarray:
    """jitted so xla fuses sample/scale/cast and never lands the f32 tensor in hbm"""

    return (jax.random.normal(key, shape, jnp.float32) / np.sqrt(fan_in)).astype(DTYPE)


def make_inputs(m: int, d: int, f: int, e: int, skew: float = 0.0, seed: int = 0):
    """rows sorted by expert id, as the router and sort would hand them over"""

    keys = jax.random.split(jax.random.key(seed), 6)

    logits = jax.random.normal(keys[0], (e,)) * skew  # skew > 0 imbalances the experts
    ids = jnp.sort(jax.random.categorical(keys[1], logits, shape=(m,)))
    group_sizes = jnp.bincount(ids, length=e).astype(jnp.int32)

    lhs = jax.random.normal(keys[2], (m, d), DTYPE)
    params = {
        "up": normal(keys[3], (e, d, f), d),
        "gate": normal(keys[4], (e, d, f), d),
        "down": normal(keys[5], (e, f, d), f),
    }
    return lhs, params, group_sizes


def ref_expert_ffn(lhs: jnp.ndarray, params: dict, group_sizes: jnp.ndarray) -> jnp.ndarray:
    """swiglu expert mlp as three ragged dots"""

    rdot = functools.partial(
        jax.lax.ragged_dot, group_sizes=group_sizes, preferred_element_type=DTYPE
    )
    act = rdot(lhs, params["up"])
    gate = jax.nn.swish(rdot(lhs, params["gate"]))
    return rdot(gate * act, params["down"])

In [3]:
import timeit


def bench(fn, *args, ntrials: int = 50) -> float:
    """benchmark given function"""

    fn = jax.jit(fn)
    jax.block_until_ready(fn(*args))
    result = timeit.timeit(lambda: jax.block_until_ready(fn(*args)), number=ntrials)
    return result / ntrials


def check(y: jnp.ndarray, y_ref: jnp.ndarray, rtol: float = 2e-2, atol: float = 2e-2) -> bool:
    """bf16 accumulation makes the default tolerances useless, so compare in f32"""

    y, y_ref = np.asarray(y, np.float32), np.asarray(y_ref, np.float32)
    return bool(np.all(np.abs(y - y_ref) <= atol + rtol * np.abs(y_ref)))


def profile(
    fn, *args, logdir: str = "/tmp/tensorboard", trace_mode: str = "TRACE_COMPUTE_AND_SYNC"
):
    """one trace per call, each landing as its own run in the profile tab"""

    options = jax.profiler.ProfileOptions()
    options.advanced_configuration = {"tpu_trace_mode": trace_mode}

    fn = jax.jit(fn)
    with jax.profiler.trace(logdir, profiler_options=options):
        out = fn(*args)
        jax.block_until_ready(out)

In [4]:
M, D, F, E = 16384, 1024, 4096, 8

x, params, group_sizes = make_inputs(M, D, F, E)
secs = bench(ref_expert_ffn, x, params, group_sizes)
print(f"ref {secs * 1e3:.3f} ms  {6 * M * D * F / secs / 1e12:.1f} TFLOP/s")

ref 4.902 ms  84.1 TFLOP/s


In [5]:
profile(ref_expert_ffn, x, params, group_sizes)

2026-07-31 06:54:48.432995: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-31 06:54:48.445429: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [6]:
%load_ext tensorboard

In [7]:
%tensorboard --logdir /tmp/tensorboard

- `ragged-dot-none.1` (1553us) : act = rdot(lhs, params["up"])
- `ragged-dot-none` (1136us) : gate = jax.nn.swish(rdot(lhs, params["gate"]))
- `multiply_multiply_fusion` (946us) : temp = gate * act
- `ragged-dot-none.2` (1091us) : rdot(temp, params["down"])

->  Fuse the up/down projections into one so there is no useless communication between ops!

## The kernel

```
y[i] = down[g(i)] @ (swish(x[i] @ gate[g(i)]) * (x[i] @ up[g(i)]))
```

`lhs` is `(m, d)` bf16 sorted by expert, `up`/`gate` are `(e, d, f)`, `down` is `(e, f, d)`,
`group_sizes` is `(e,)` int32 summing to `m`. Return `(m, d)` bf16 and run under `jit` with
`m, d, f, e` static. Tuning knobs go in `**kwargs`; `compare` forwards them.

In [8]:
def fused_expert_ffn_kernel(
    x_hbm,
    up_hbm,
    gate_hbm,
    down_hbm,
    group_map_hbm,
    msteps,
    out_hbm,
    group_map_smem,
    up_acc_scratch,
    gate_acc_scratch,
    acc_scratch,
    *,
    isteps: int,
    jsteps: int,
    ksteps: int,
    blk_m: int,
    blk_i: int,
    blk_j: int,
    blk_k: int,
):

    pltpu.sync_copy(group_map_hbm, group_map_smem)

    def pipeline_body(x_vmem, up_vmem, gate_vmem, down_vmem, o_vmem):
        iidx, jidx, kidx = pl.program_id(1), pl.program_id(2), pl.program_id(3)

        # Reset output scratch
        @pl.when((iidx == 0) & (jidx == 0) & (kidx == 0))
        def _():
            acc_scratch[...] = jnp.zeros_like(acc_scratch)

        # Reset only when we are beginning a new (m, i) calculation
        @pl.when((jidx == 0) & (kidx == 0))
        def _():
            up_acc_scratch[...] = jnp.zeros_like(up_acc_scratch)
            gate_acc_scratch[...] = jnp.zeros_like(gate_acc_scratch)

        # Do matmul only once per j
        @pl.when(jidx == 0)
        def _():
            up_acc_scratch[...] += jnp.dot(
                x_vmem[...], up_vmem[...].squeeze(), preferred_element_type=jnp.float32
            )
            gate_acc_scratch[...] += jnp.dot(
                x_vmem[...], gate_vmem[...].squeeze(), preferred_element_type=jnp.float32
            )

        # Accumulate
        @pl.when(kidx == ksteps - 1)
        def _():
            partial_sum = jnp.dot(
                up_acc_scratch[...] * jax.nn.swish(gate_acc_scratch[...]),
                down_vmem[...].squeeze(),
                preferred_element_type=jnp.float32,
            )
            acc_scratch[:, pl.ds(jidx * blk_j, blk_j)] += partial_sum[...]

        @pl.when((iidx == isteps - 1) & (kidx == ksteps - 1))
        def _():
            o_vmem[...] = acc_scratch[:, pl.ds(jidx * blk_j, blk_j)].astype(o_vmem.dtype)

    def x_map(m, i, j, k):
        del i, j
        return (m, k)

    def up_map(m, i, j, k):
        del j
        return (group_map_smem[m], k, i)

    def gate_map(m, i, j, k):
        del j
        return (group_map_smem[m], k, i)

    def down_map(m, i, j, k):
        del k
        return (group_map_smem[m], i, j)

    def o_map(m, i, j, k):
        del i, k
        return (m, j)

    x_spec = pl.BlockSpec(block_shape=(blk_m, blk_k), index_map=x_map)
    up_spec = pl.BlockSpec(block_shape=(1, blk_k, blk_i), index_map=up_map)
    gate_spec = pl.BlockSpec(block_shape=(1, blk_k, blk_i), index_map=gate_map)
    down_spec = pl.BlockSpec(block_shape=(1, blk_i, blk_j), index_map=down_map)
    o_spec = pl.BlockSpec(block_shape=(blk_m, blk_j), index_map=o_map)

    pltpu.emit_pipeline(
        pipeline_body,
        grid=(msteps[0], isteps, jsteps, ksteps),
        in_specs=[x_spec, up_spec, gate_spec, down_spec],
        out_specs=o_spec,
    )(x_hbm, up_hbm, gate_hbm, down_hbm, out_hbm)


"""
y[m, j] = \sum_i ((\sum_k x[m, k] @ up_e[k, i]) * swish(\sum_k x[m, k] @ gate_e[k, i])) @ down_e[i, j]

for m in (0, slices):
    for i in (0, f):
        for j in (0, d):
            for k in (0, d):
                if j == 0 and k == 0:
                    a = 0
                    g = 0
                if j == 0:
                    a += x[m, k] @ up[e[m], k, i]  ----> a[m, i]
                    g += x[m, k] @ gate[e[m], k, i]  ----> g[m, i]
                if k == d - 1:
                    acc[m, j] += a * swish(g) @ down[e[m], i, j]  ----> y[m, j]
"""


def get_pad_info(group_sizes: jnp.ndarray, block: int, rows: int):
    group_starts = jnp.cumsum(group_sizes) - group_sizes
    padded = -(-group_sizes // block) * block  # ceil division
    pad_group_starts = jnp.cumsum(padded) - padded

    max_size = rows + group_sizes.shape[0] * block
    slot = jnp.arange(max_size)
    group_map = jnp.searchsorted(pad_group_starts, slot, side="right") - 1
    row_in_group = slot - pad_group_starts[group_map]
    is_valid = row_in_group < group_sizes[group_map]
    src_row = group_starts[group_map] + row_in_group

    return group_starts, pad_group_starts, group_map, src_row, is_valid


def pad_array(x: jnp.ndarray, group_sizes: jnp.ndarray, block: int, rows: int):
    _, _, group_map, src_row, is_valid = get_pad_info(group_sizes, block, rows)
    return jnp.where(is_valid[:, None], x[src_row], 0), group_map


def unpad_array(x: jnp.ndarray, group_sizes: jnp.ndarray, block: int, rows: int):
    group_starts = jnp.cumsum(group_sizes) - group_sizes
    padded = -(-group_sizes // block) * block  # ceil division
    pad_group_starts = jnp.cumsum(padded) - padded

    row = jnp.arange(rows)
    group_map = jnp.searchsorted(group_starts, row, side="right") - 1
    return x[pad_group_starts[group_map] + (row - group_starts[group_map])]


def fused_expert_ffn(
    x: jnp.ndarray,
    params: dict,
    group_sizes: jnp.ndarray,
    *,
    bm: int = 128,
    bi: int = 1024,
    bj: int = 512,
    bk: int = 512,
):
    m, _ = x.shape
    _, d, f = params["up"].shape

    msteps = -(-group_sizes // bm).sum()[None]
    pad_x, group_map = pad_array(x, group_sizes, bm, m)
    tile_group_map = group_map[::bm]

    hbm_block_spec = pl.BlockSpec(memory_space=pl.ANY)
    temp = pl.pallas_call(
        functools.partial(
            fused_expert_ffn_kernel,
            isteps=f // bi,
            jsteps=d // bj,
            ksteps=d // bk,
            blk_m=bm,
            blk_i=bi,
            blk_j=bj,
            blk_k=bk,
        ),
        in_specs=[
            hbm_block_spec,
            hbm_block_spec,
            hbm_block_spec,
            hbm_block_spec,
            hbm_block_spec,
            pl.BlockSpec(memory_space=pltpu.SMEM),
        ],
        out_specs=hbm_block_spec,
        out_shape=jax.ShapeDtypeStruct(pad_x.shape, pad_x.dtype),
        scratch_shapes=[
            pltpu.SMEM(tile_group_map.shape, jnp.int32),
            pltpu.VMEM((bm, bi), jnp.float32),
            pltpu.VMEM((bm, bi), jnp.float32),
            pltpu.VMEM((bm, d), jnp.float32),
        ],
    )(pad_x, params["up"], params["gate"], params["down"], tile_group_map, msteps)

    return unpad_array(temp, group_sizes, bm, m)


In [9]:
def compare(m: int, d: int, f: int, e: int, skew: float = 0.0, ntrials: int = 50, **kwargs) -> dict:
    """checks the kernel against the reference and times both at the same shape"""

    lhs, params, group_sizes = make_inputs(m, d, f, e, skew)
    fused = functools.partial(fused_expert_ffn, **kwargs)

    y_ref = jax.jit(ref_expert_ffn)(lhs, params, group_sizes)
    y = jax.jit(fused, static_argnames=["bm", "bi", "bj", "bk"])(lhs, params, group_sizes)

    ref_secs = bench(ref_expert_ffn, lhs, params, group_sizes, ntrials=ntrials)
    secs = bench(fused, lhs, params, group_sizes, ntrials=ntrials)

    return {
        "m": m,
        "d": d,
        "f": f,
        "e": e,
        "skew": skew,
        "f/d": f / d,
        "ok": check(y, y_ref),
        "ref_ms": ref_secs * 1e3,
        "fused_ms": secs * 1e3,
        "speedup": ref_secs / secs,
        "TFLOP/s": 6 * m * d * f / secs / 1e12,
    }


compare(M, D, F, E, bm=256, bi=1024, bj=1024, bk=512)

{'m': 16384,
 'd': 1024,
 'f': 4096,
 'e': 8,
 'skew': 0.0,
 'f/d': 4.0,
 'ok': True,
 'ref_ms': 4.904326340001717,
 'fused_ms': 3.084986839967314,
 'speedup': 1.5897397928782344,
 'TFLOP/s': 133.65271289791582}

In [ ]:
def blocks(m: int, d: int, f: int, e: int) -> dict:
    """block sizes derived from the shape, so every sweep tunes the same way"""

    return {
        "bm": min(256, m // (e * 2)),
        "bi": min(1024, f // 4),
        "bj": min(1024, d),
        "bk": min(512, d // 2),
    }


def pow2_labels(xs):
    """ticks as 2^k, so the long ones (131072) still fit at page width"""

    return [rf"$2^{{{int(round(np.log2(v)))}}}$" for v in xs]

## Sweeps

One variable doubles per point, the rest held at `(m, d, f, e) = (16384, 1024, 4096, 8)`.
`d` starts at 256 because `bk = d // 2` has to stay a multiple of 128, and `f` starts one
doubling above `d` to keep `f > d`.

In [19]:
def sweep(shapes: list[tuple[int, int, int, int]], ntrials: int = 10) -> pd.DataFrame:
    """time the kernel against the reference over a list of (m, d, f, e)"""

    return pd.DataFrame([compare(*shape, ntrials=ntrials, **blocks(*shape)) for shape in shapes])


SWEEPS = {
    # more experts over the same tokens: groups shrink, tiles straddle boundaries
    "e": [(16384, 1024, 4096, e) for e in [1, 2, 4, 8, 16, 32, 64, 128]],
    # more tokens at a fixed 2048 rows per expert: pure weak scaling
    "m": [(m, 1024, 4096, m // 2048) for m in [2048, 4096, 8192, 16384, 32768, 65536, 131072]],
    # wider ffn at fixed model dim: the f/d ratio the fusion is supposed to exploit
    "f": [(16384, 1024, f, 8) for f in [2048, 4096, 8192, 16384, 32768]],
    # bigger model at a fixed f/d of 4: shape stays similar, arithmetic intensity grows
    "d": [(16384, d, 4 * d, 8) for d in [256, 512, 1024, 2048]],  # 4096 wants >16MB vmem
}

results = {name: sweep(shapes) for name, shapes in SWEEPS.items()}

In [ ]:
TITLES = {
    "e": "experts, m fixed",
    "m": "tokens, m/e fixed",
    "f": "ffn dim, d fixed",
    "d": "model dim, f/d fixed",
}

fig, axes = plt.subplots(2, 4, figsize=(7.5, 4.0), sharey="row")

for col, (name, df) in enumerate(results.items()):
    xs, top, bot = df[name], axes[0, col], axes[1, col]

    top.plot(xs, df["TFLOP/s"] / df.speedup, "o-", label="ragged_dot")
    top.plot(xs, df["TFLOP/s"], "o-", label="fused")
    top.set_title(f"({'abcd'[col]}) {TITLES[name]}", fontsize=8)

    bot.plot(xs, df.speedup, "o-", color="tab:green")
    bot.axhline(1.0, color="gray", ls="--", lw=0.6)
    bot.set_xlabel(name)

    mismatch = ~df.ok
    if mismatch.any():  # a fast wrong answer is not a win
        bot.plot(xs[mismatch], df.speedup[mismatch], "rx", ms=12, label="mismatch")
        bot.legend()

    for ax in (top, bot):
        ax.set_xscale("log", base=2)
        ax.set_xticks(xs)
        ax.set_xticklabels(pow2_labels(xs), fontsize=7)
        ax.tick_params(axis="y", labelsize=7)
        ax.grid(alpha=0.3)

axes[0, 0].set_ylabel("TFLOP/s", fontsize=8)
axes[0, 0].legend(fontsize=7)
axes[1, 0].set_ylabel("speedup vs ragged_dot", fontsize=8)
fig.tight_layout()
os.makedirs("plots", exist_ok=True)
fig.savefig("plots/moe_shape_sweeps.png", dpi=200, bbox_inches="tight")

## Skew

Everything above runs on perfectly balanced experts, which is the one routing distribution real MoE
never produces. `skew` scales the logits `make_inputs` samples expert ids from, so larger values
concentrate tokens on fewer experts.

`skew` itself is an arbitrary knob, so the three panels read left to right as cause and effect: how
much speedup you get, how lopsided the groups actually became, and how many experts ended up with no
tokens at all. Each line is one shape, and the legend names the two things that differ between
them -- the expert count and the f/d ratio. All three are m=16384, d=1024.

In [21]:
def group_stats(m: int, e: int, skew: float, seed: int = 0) -> dict:
    """what a given skew actually does to the group sizes, so the x axis means something"""

    keys = jax.random.split(jax.random.key(seed), 6)
    logits = jax.random.normal(keys[0], (e,)) * skew
    ids = jnp.sort(jax.random.categorical(keys[1], logits, shape=(m,)))
    sizes = np.asarray(jnp.bincount(ids, length=e))
    return {"empty": int((sizes == 0).sum()), "max/mean": float(sizes.max() / sizes.mean())}


SKEWS = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]
SKEW_SHAPES = {
    r"$e=8$, $f/d=4$": (16384, 1024, 4096, 8),
    r"$e=64$, $f/d=4$": (16384, 1024, 4096, 64),
    r"$e=8$, $f/d=8$": (16384, 1024, 8192, 8),
}

skew_results = {
    name: pd.DataFrame(
        [
            compare(*shape, skew=s, ntrials=10, **blocks(*shape))
            | group_stats(shape[0], shape[3], s)
            for s in SKEWS
        ]
    )
    for name, shape in SKEW_SHAPES.items()
}

In [ ]:
fig, (gain, imbalance, empty) = plt.subplots(1, 3, figsize=(7.5, 2.4))

for name, df in skew_results.items():
    gain.plot(df["skew"], df.speedup, "o-", label=name)
    imbalance.plot(df["skew"], df["max/mean"], "o-")
    empty.plot(df["skew"], df["empty"], "o-")

gain.axhline(1.0, color="gray", ls="--", lw=0.6)
gain.set_title("(a) kernel speedup", fontsize=8)
gain.set_ylabel("fused / ragged_dot", fontsize=8)
gain.legend(fontsize=7)

imbalance.set_title("(b) imbalance the skew produced", fontsize=8)
imbalance.set_ylabel("largest group / mean group", fontsize=8)

empty.set_title("(c) experts left with no tokens", fontsize=8)
empty.set_ylabel("count", fontsize=8)

for ax in (gain, imbalance, empty):
    ax.set_xlabel("skew")
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.3)

fig.tight_layout()
os.makedirs("plots", exist_ok=True)
fig.savefig("plots/moe_skew.png", dpi=200, bbox_inches="tight")

## f/d at constant FLOPs

The `f` sweep raises the ratio and the total work together, so "speedup grows with f" could just be
"speedup grows with work". Holding `d * f` fixed separates them: all three points below are the same
6*m*d*f, so the rates are directly comparable and the only thing changing is the shape of the ffn.
Three points is all that `f > d` and the 128-multiple rule on `bk` allow.

In [ ]:
RATIOS = [(16384, d, 2**22 // d, 8) for d in [256, 512, 1024]]  # f/d = 64, 16, 4

ratios = sweep(RATIOS)

In [ ]:
xs = ratios["f"] / ratios["d"]

fig, (rate, gain) = plt.subplots(1, 2, figsize=(5.6, 2.4))

rate.plot(xs, ratios["TFLOP/s"] / ratios.speedup, "o-", label="ragged_dot")
rate.plot(xs, ratios["TFLOP/s"], "o-", label="fused")
rate.set_title("(a) throughput at equal FLOPs", fontsize=8)
rate.set_ylabel("TFLOP/s", fontsize=8)
rate.legend(fontsize=7)

gain.plot(xs, ratios.speedup, "o-", color="tab:green")
gain.axhline(1.0, color="gray", ls="--", lw=0.6)
gain.set_title("(b) kernel speedup", fontsize=8)
gain.set_ylabel("fused / ragged_dot", fontsize=8)

for ax in (rate, gain):
    ax.set_xscale("log", base=2)
    ax.set_xticks(xs)
    ax.set_xticklabels(pow2_labels(xs), fontsize=7)
    ax.tick_params(axis="y", labelsize=7)
    ax.set_xlabel("f/d")
    ax.grid(alpha=0.3)

fig.tight_layout()
os.makedirs("plots", exist_ok=True)
fig.savefig("plots/moe_ratio.png", dpi=200, bbox_inches="tight")